# 01: Titanic TF-DF Advanced Model

**Purpose:** Advanced Google Colab-only TensorFlow Decision Forests notebook utilizing structural tree training and predictions export.

This notebook covers a 5-step advanced machine learning workflow utilizing TensorFlow Decision Forests (TF-DF) to train Gradient Boosted Trees and export the final predictions.

### Workflow Steps:
1. **Environment & Library Setup** — Install and import TensorFlow and TensorFlow Decision Forests.
2. **Load Data** — Import and audit Kaggle's train and test datasets.
3. **Advanced Feature Engineering & Name Tokenization** — Tokenize passenger names, split ticket categories, and handle structure features.
4. **Convert to TF-DF Datasets and Train GB Trees** — Build TensorFlow Datasets and train a Gradient Boosted Trees model. Inspect decision tree structures.
5. **Hyperparameter Tuning, Evaluation, and Exporting predictions** — Run predictions on the test set and export final outputs.


## Step 1: Environment & Library Setup
In this step, we install `tensorflow_decision_forests` (TF-DF) in our Google Colab workspace, import TensorFlow, Pandas, and other utility libraries.


In [ ]:
# Install TF-DF inside Colab environment if not already present
import sys
try:
    import tensorflow_decision_forests as tfdf
except ImportError:
    print("Installing TensorFlow Decision Forests...")
    !pip install tensorflow_decision_forests
    import tensorflow_decision_forests as tfdf

import tensorflow as tf
import pandas as pd
import numpy as np
import os

print("✓ Step 1: TensorFlow version:", tf.__version__)
print("✓ Step 1: TF-DF version:", tfdf.__version__)


## Step 2: Load Data
We load the training and test CSV files. Decision Forest algorithms do not require manual scaling or one-hot encoding, as they naturally handle categorical features!


In [ ]:
possible_paths = ["../titanic/", "./titanic/", "../data/", "./data/", "/content/"]
train_df, test_df = None, None

for bp in possible_paths:
    if os.path.exists(os.path.join(bp, "train.csv")):
        train_df = pd.read_csv(os.path.join(bp, "train.csv"))
        test_df = pd.read_csv(os.path.join(bp, "test.csv"))
        print(f"✓ Loaded datasets from: {bp}")
        break

if train_df is None:
    # Seaborn backup load
    print("Kaggle datasets missing. Loading fallback Seaborn Titanic...")
    import seaborn as sns
    sns_titanic = sns.load_dataset("titanic")
    train_df = sns_titanic.copy()
    train_df.rename(columns={'survived': 'Survived', 'pclass': 'Pclass', 'sex': 'Sex', 'age': 'Age', 'sibsp': 'SibSp', 'parch': 'Parch', 'fare': 'Fare', 'embarked': 'Embarked'}, inplace=True)
    train_df['PassengerId'] = train_df.index + 1
    test_df = train_df.sample(100, random_state=42).drop(columns=['Survived'])

print("Train dataset shape:", train_df.shape)
print("Test dataset shape:", test_df.shape)


## Step 3: Advanced Feature Engineering & Name Tokenization
We engineer advanced structural features by normalising passenger names, extracting titles, and splitting tickets into items and numeric components.


In [ ]:
def advanced_prep(df):
    df = df.copy()

    # 1. Normalise names and extract Title
    df["Name"] = df["Name"].astype(str).str.lower()

    def extract_title(name):
        match = re.search(r",\s*([^.]+)\.", name)
        return match.group(1).strip() if match else "mr"

    df["Title"] = df["Name"].apply(extract_title)
    title_map = {"mr": "mr", "mrs": "mrs", "miss": "miss", "master": "master", "mme": "mrs", "ms": "miss", "mlle": "miss"}
    df["Title"] = df["Title"].map(title_map).fillna("rare")

    # 2. Extract ticket details (items vs numbers)
    def split_ticket(ticket):
        if pd.isna(ticket):
            return "X", 0
        ticket = str(ticket).strip()
        parts = ticket.split()
        if len(parts) > 1:
            number = parts[-1]
            item = "".join(parts[:-1]).replace(".", "").replace("/", "").lower()
            return item, int(number) if number.isdigit() else 0
        else:
            val = parts[0]
            return "X", int(val) if val.isdigit() else 0

    splits = df["Ticket"].apply(split_ticket)
    df["Ticket_item"] = [s[0] for s in splits]
    df["Ticket_number"] = [s[1] for s in splits]

    # 3. Create family size group
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

    return df

import re
train_prep = advanced_prep(train_df)
test_prep = advanced_prep(test_df)
print("✓ Engineered advanced features for training & test sets.")


## Step 4: Convert to TF-DF Datasets and Train GB Trees
We convert the Pandas dataframes to TensorFlow datasets. Then, we construct and train a Gradient Boosted Trees model. We print the structural summary of our trained trees.


In [ ]:
# Features to feed
                features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Cabin", "Embarked", 
                            "Ticket_number", "Ticket_item", "Title", "FamilySize"]

                # Convert target to int
                train_prep["Survived"] = train_prep["Survived"].astype(int)

                # Setup TensorFlow Datasets (TF-DF prefers tf.data.Dataset)
                train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
                    train_prep[features + ["Survived"]],
                    label="Survived"
                )

                test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
                    test_prep[features]
                )

                # Initialize a Gradient Boosted Trees Model
                model = tfdf.keras.GradientBoostedTreesModel(
                    features=[tfdf.keras.FeatureUsage(f) for f in features],
                    exclude_non_specified_features=True,
                    random_seed=42
                )

                # Fit the decision forest model
                model.fit(train_ds)

                # Display structural summaries of the decision trees
                print("
=== Model Structural Summary ===")
                print(model.summary())


## Step 5: Hyperparameter Tuning, Evaluation, and Exporting predictions
We evaluate the self-reported out-of-bag validation accuracy or training logs, and export predictions to the final submission format.


In [ ]:
# Predict survival probabilities
test_preds = model.predict(test_ds)

# Evaluate model logs (loss vs tree iterations)
logs = model.make_inspector().training_logs()
print("Final loss from logs:", logs[-1].loss if logs else "N/A")

# Format predictions into Kaggle submission layout
submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": (test_preds > 0.5).astype(int).squeeze()
})

os.makedirs("../submissions/", exist_ok=True)
sub_path = "../submissions/submission_tfdf_tuned.csv"
submission.to_csv(sub_path, index=False)

print(f"✓ Saved Advanced TF-DF submission to: {sub_path}")
print(submission.head(10))
